# Ensemble Methods — Bagging and Random Forests

**Ensembles** combine many models so that their individual errors cancel out.

### Bagging (Bootstrap AGGregatING)
1. Draw $B$ **bootstrap samples** (sample $n$ points *with replacement*).
2. Train one high-variance model (e.g. an unpruned decision tree) on each.
3. Predict by **majority vote** / averaging probabilities (classification) or averaging (regression).

Averaging $B$ models with variance $\sigma^2$ and pairwise correlation $\rho$ gives variance
$$\rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$$
so bagging reduces variance, but only down to $\rho\sigma^2$.

### Random Forest
Bagging of trees **plus** a random subset of $m$ features considered at every split (typically $m=\sqrt{d}$).
This **decorrelates** the trees (smaller $\rho$) so averaging helps more.

### Out-of-bag (OOB) evaluation
Each bootstrap sample leaves out ≈ 36.8 % of the data ($ (1-1/n)^n \to e^{-1}$). Predicting each point
using only the trees that did not see it gives a free validation estimate.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

In [ ]:
def make_moons(n_samples=200, noise=0.1, random_state=None):
    """Two interleaving half circles."""
    rng = np.random.RandomState(random_state)
    n_out = n_samples // 2
    n_in = n_samples - n_out
    t_out = np.linspace(0, np.pi, n_out)
    t_in = np.linspace(0, np.pi, n_in)
    outer = np.c_[np.cos(t_out), np.sin(t_out)]
    inner = np.c_[1 - np.cos(t_in), 0.5 - np.sin(t_in)]
    X = np.vstack([outer, inner]) + rng.normal(0, noise, size=(n_samples, 2))
    y = np.r_[np.zeros(n_out, dtype=int), np.ones(n_in, dtype=int)]
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def train_test_split(X, y, test_size=0.25, random_state=None):
    """Shuffle and split arrays into train and test subsets."""
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(X))
    n_test = int(round(len(X) * test_size))
    test, train = idx[:n_test], idx[n_test:]
    return X[train], X[test], y[train], y[test]

def accuracy_score(y_true, y_pred):
    return np.mean(np.asarray(y_true) == np.asarray(y_pred))

def plot_decision_boundary(model, X, y, title="", ax=None, resolution=150):
    """Colour the plane by the model's prediction and overlay the data."""
    ax = ax or plt.gca()
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))
    Z = np.asarray(model.predict(np.c_[xx.ravel(), yy.ravel()]), dtype=float).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=25)
    ax.set_title(title)

## Base learner: a compact CART decision tree

In [ ]:
class Node:
    def __init__(self, value):
        self.value = value                     # class-probability vector or regression value
        self.feature = self.threshold = self.left = self.right = None
        self.leaf_id = None


class DecisionTree:
    """Compact CART (see 06_Decision_Tree_CART) used as the base learner.

    Classification labels must be integers 0..K-1; leaves store class probabilities.
    max_features: None (all), int, or "sqrt" -> random feature subset at every split (Random Forest).
    """

    def __init__(self, task="classification", max_depth=None, min_samples_split=2,
                 min_samples_leaf=1, max_features=None, random_state=None):
        self.task = task
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.rng = np.random.RandomState(random_state)

    def fit(self, X, y, n_classes=None):
        X = np.asarray(X, dtype=float)
        if self.task == "classification":
            y = np.asarray(y, dtype=int)
            self.n_classes = n_classes or y.max() + 1
        else:
            y = np.asarray(y, dtype=float)
        d = X.shape[1]
        if self.max_features == "sqrt":
            self.k_features = max(1, int(np.sqrt(d)))
        else:
            self.k_features = d if self.max_features is None else self.max_features
        self.feature_importances_ = np.zeros(d)
        self.leaves_ = []
        self.root = self._build(X, y, 0)
        if self.feature_importances_.sum() > 0:
            self.feature_importances_ /= self.feature_importances_.sum()
        return self

    def _impurity(self, y):
        if self.task == "classification":
            p = np.bincount(y, minlength=self.n_classes) / len(y)
            return 1 - np.sum(p ** 2)
        return y.var()

    def _leaf(self, node):
        node.leaf_id = len(self.leaves_)
        self.leaves_.append(node)
        return node

    def _best_split(self, X, y):
        n, d = X.shape
        features = self.rng.choice(d, self.k_features, replace=False)
        nl = np.arange(1, n)
        nr = n - nl
        best = (None, None, np.inf)
        for f in features:
            order = np.argsort(X[:, f], kind="mergesort")
            xs, ys = X[order, f], y[order]
            if self.task == "classification":
                onehot = np.eye(self.n_classes)[ys]
                left = np.cumsum(onehot, axis=0)[:-1]
                right = onehot.sum(axis=0) - left
                cost = (nl * (1 - np.sum((left / nl[:, None]) ** 2, 1))
                        + nr * (1 - np.sum((right / nr[:, None]) ** 2, 1))) / n
            else:
                cs, cs2 = np.cumsum(ys)[:-1], np.cumsum(ys ** 2)[:-1]
                cost = ((cs2 - cs ** 2 / nl) + ((ys ** 2).sum() - cs2) - (ys.sum() - cs) ** 2 / nr) / n
            valid = (xs[1:] != xs[:-1]) & (nl >= self.min_samples_leaf) & (nr >= self.min_samples_leaf)
            if valid.any():
                cost = np.where(valid, cost, np.inf)
                i = np.argmin(cost)
                if cost[i] < best[2]:
                    best = (f, (xs[i] + xs[i + 1]) / 2, cost[i])
        return best

    def _build(self, X, y, depth):
        if self.task == "classification":
            node = Node(np.bincount(y, minlength=self.n_classes) / len(y))
        else:
            node = Node(y.mean())
        impurity = self._impurity(y)
        if (impurity < 1e-12 or len(y) < self.min_samples_split
                or (self.max_depth is not None and depth >= self.max_depth)):
            return self._leaf(node)
        f, t, cost = self._best_split(X, y)
        if f is None:
            return self._leaf(node)
        self.feature_importances_[f] += len(y) * (impurity - cost)    # weighted impurity decrease
        mask = X[:, f] <= t
        node.feature, node.threshold = f, t
        node.left = self._build(X[mask], y[mask], depth + 1)
        node.right = self._build(X[~mask], y[~mask], depth + 1)
        return node

    def _route(self, node, X, idx, out):
        if node.left is None:
            out[idx] = node.leaf_id
            return
        mask = X[idx, node.feature] <= node.threshold
        self._route(node.left, X, idx[mask], out)
        self._route(node.right, X, idx[~mask], out)

    def apply(self, X):
        """Index (into self.leaves_) of the leaf each sample lands in."""
        X = np.asarray(X, dtype=float)
        out = np.empty(len(X), dtype=int)
        self._route(self.root, X, np.arange(len(X)), out)
        return out

    def predict_proba(self, X):
        return np.array([leaf.value for leaf in self.leaves_])[self.apply(X)]

    def predict(self, X):
        values = self.predict_proba(X)
        return values.argmax(axis=1) if self.task == "classification" else values

## Bootstrap sampling

In [ ]:
rng = np.random.RandomState(0)
n = 1000
unique_frac = [len(np.unique(rng.randint(0, n, n))) / n for _ in range(500)]
print(f"Average fraction of unique points in a bootstrap sample: {np.mean(unique_frac):.4f}")
print(f"Theory: 1 - 1/e = {1 - np.exp(-1):.4f}")

## Bagging and Random Forest implementation

In [ ]:
class BaggingClassifier:
    def __init__(self, n_estimators=50, max_features=None, max_depth=None, min_samples_leaf=1,
                 oob_score=False, random_state=None):
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.oob_score = oob_score
        self.random_state = random_state

    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        n = len(y)
        self.n_classes = y.max() + 1
        self.trees = []
        oob_votes = np.zeros((n, self.n_classes))
        for _ in range(self.n_estimators):
            idx = rng.randint(0, n, n)                               # bootstrap sample
            tree = DecisionTree(max_depth=self.max_depth, min_samples_leaf=self.min_samples_leaf,
                                max_features=self.max_features, random_state=rng.randint(1 << 30))
            tree.fit(X[idx], y[idx], n_classes=self.n_classes)
            self.trees.append(tree)
            if self.oob_score:
                oob = np.ones(n, dtype=bool)
                oob[idx] = False
                oob_votes[oob] += tree.predict_proba(X[oob])
        if self.oob_score:
            seen = oob_votes.sum(axis=1) > 0
            self.oob_score_ = np.mean(oob_votes[seen].argmax(axis=1) == y[seen])
        self.feature_importances_ = np.mean([t.feature_importances_ for t in self.trees], axis=0)
        return self

    def predict_proba(self, X):
        return np.mean([t.predict_proba(X) for t in self.trees], axis=0)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

    def staged_predict(self, X):
        """Predictions using the first 1, 2, ..., B trees."""
        cumulative = np.cumsum([t.predict_proba(X) for t in self.trees], axis=0)
        return cumulative.argmax(axis=2)


class RandomForestClassifier(BaggingClassifier):
    """Bagging + random feature subset at every split."""

    def __init__(self, n_estimators=100, max_features="sqrt", **kwargs):
        super().__init__(n_estimators=n_estimators, max_features=max_features, **kwargs)

## Decision boundaries: single tree vs bagging vs random forest

In [ ]:
X, y = make_moons(n_samples=500, noise=0.35, random_state=4)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

models = {
    "Single tree": DecisionTree(random_state=0).fit(X_train, y_train),
    "Bagging (100 trees)": BaggingClassifier(100, random_state=0).fit(X_train, y_train),
    "Random forest (100 trees, m=1)": RandomForestClassifier(100, max_features=1, random_state=0).fit(X_train, y_train),
}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, (name, m) in zip(axes, models.items()):
    plot_decision_boundary(m, X_train, y_train, ax=ax,
                           title=f"{name}\ntest acc = {accuracy_score(y_test, m.predict(X_test)):.3f}")
plt.tight_layout(); plt.show()

## A higher-dimensional problem
10 features: the label depends on $x_0 \dots x_5$ via $x_0 + x_1 + x_2 - x_3 + x_4x_5 > 0$; $x_6 \dots x_9$ are pure noise.
10 % of labels are flipped.

In [ ]:
rng = np.random.RandomState(0)
Xh = rng.normal(size=(1000, 10))
yh = (Xh[:, 0] + Xh[:, 1] + Xh[:, 2] - Xh[:, 3] + Xh[:, 4] * Xh[:, 5] > 0).astype(int)
flip = rng.rand(len(yh)) < 0.1
yh[flip] = 1 - yh[flip]
Xh_train, Xh_test, yh_train, yh_test = train_test_split(Xh, yh, test_size=0.3, random_state=0)

tree = DecisionTree(random_state=0).fit(Xh_train, yh_train)
bag = BaggingClassifier(200, oob_score=True, random_state=0).fit(Xh_train, yh_train)
rf = RandomForestClassifier(200, oob_score=True, random_state=0).fit(Xh_train, yh_train)

print(f"Single tree   test acc: {accuracy_score(yh_test, tree.predict(Xh_test)):.3f}")
print(f"Bagging       test acc: {accuracy_score(yh_test, bag.predict(Xh_test)):.3f}   OOB acc: {bag.oob_score_:.3f}")
print(f"Random forest test acc: {accuracy_score(yh_test, rf.predict(Xh_test)):.3f}   OOB acc: {rf.oob_score_:.3f}")

### Accuracy as trees are added

In [ ]:
B = np.arange(1, 201)
plt.plot(B, [accuracy_score(yh_test, p) for p in bag.staged_predict(Xh_test)], label="Bagging")
plt.plot(B, [accuracy_score(yh_test, p) for p in rf.staged_predict(Xh_test)], label="Random forest")
plt.axhline(accuracy_score(yh_test, tree.predict(Xh_test)), ls="--", c="gray", label="single tree")
plt.xlabel("number of trees"); plt.ylabel("test accuracy"); plt.legend(); plt.title("Ensemble size vs accuracy"); plt.show()

### Tree correlation
Random feature selection makes the trees disagree more, which is exactly what lets averaging reduce variance.

In [ ]:
def mean_pairwise_agreement(model, X):
    preds = np.array([t.predict(X) for t in model.trees[:50]])
    agree = [(preds[i] == preds[j]).mean() for i in range(len(preds)) for j in range(i + 1, len(preds))]
    return np.mean(agree)

print(f"Mean pairwise agreement between trees — bagging: {mean_pairwise_agreement(bag, Xh_test):.3f}, "
      f"random forest: {mean_pairwise_agreement(rf, Xh_test):.3f}")

### Feature importance (mean impurity decrease)

In [ ]:
plt.bar(range(10), rf.feature_importances_, color=["tab:red"] * 6 + ["tab:gray"] * 4)
plt.xticks(range(10), [f"x{i}" for i in range(10)])
plt.ylabel("importance"); plt.title("Random forest feature importances (red = informative)"); plt.show()

## Summary
* Bagging reduces **variance** by averaging models trained on bootstrap samples.
* Random forests add per-split feature sampling, decorrelating trees for an extra variance reduction.
* More trees never hurts accuracy (it just plateaus); OOB error gives free validation.
* Feature importances fall out of the training procedure.